# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the ordered logistic regression FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and made available via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not present
!pip install -U mlcroissant

## 1. Data Loading
Let's load the dataset metadata and examine the dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Dataset metadata (use attribute access)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List all available record sets in the dataset, their IDs, and their fields.
For each record set, we print its `@id`, name, and available fields with their `@id`s.

In [ ]:
# View available record sets by `@id`
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets are defined in the Croissant schema. Please check the dataset documentation for loading records.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id} | Name: {getattr(rs, 'name', '(no name)')}")
        print("  Fields:")
        for fld in rs.fields:
            print(f"    Field @id: {fld.id} | Name: {getattr(fld, 'name', '')}")
        print()

# For exploration below, collect all record set @ids
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

## 3. Data Extraction
Extract records from each record set into a pandas DataFrame for analysis.
We use only the record sets discovered above, referencing each by its `@id`.

In [ ]:
# This notebook loads all available record sets (if any) into dataframes indexed by their @id
dataframes = {}

if not record_set_ids:
    print("No record sets detected to extract records from. Please refer to dataset documentation or explore the `distributions` in metadata.")
else:
    for rs_id in record_set_ids:
        # Each record returned is a dict
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
        else:
            print(f"No records loaded for RecordSet @id: {rs_id}")
    # Display the columns of the first DataFrame loaded
    if dataframes:
        first_rs = list(dataframes.keys())[0]
        print(f"\nColumns in DataFrame for RecordSet @id {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA on one loaded record set and its numeric field(s): filtering, normalization, and grouping.

_Instructions:_
- Select a numeric field by its `@id` from the previous overview
- Filter based on a threshold, normalize, and group by a categorical column (if present)
- All field names must be referenced by `@id` for consistency

If your chosen record set does not exist or contains no data, adjust accordingly.

In [ ]:
# Based on previous steps, pick a record set and a numeric field by `@id`. (Edit these as needed)
if not dataframes:
    print("No dataframes available for EDA. Please check previous steps.")
else:
    record_set_id = list(dataframes.keys())[0]  # Example: pick first available record set
    df = dataframes[record_set_id]
    print(f"Exploring RecordSet @id: {record_set_id}")
    # Try to auto-detect a numeric field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric field detected for EDA.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}\n")
        # Use a threshold value (e.g., 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)} records):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to pick a group field (categorical), excluding the numeric field
        group_field_id = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if pd.api.types.is_object_dtype(df[col]) or df[col].dtype.name == 'category':
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Show one or more basic plots for numeric fields in the examined record set.

- Histogram or KDE for numeric field
- Boxplot or categorical grouping if grouping field exists

All axes or labels should use the proper field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_field_id:
    print("No numeric data available for visualization.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field available, plot boxplot
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
- Demonstrated loading, exploration, and initial analysis of the FAIR^2 dataset using `mlcroissant`.
- All references to dataset structures use the Croissant entity `@id` (for record sets, fields, etc.).
- Next steps: you may dive further into any record set, create hypothesis tests, or derive new features from the dataset as needed.
